# 04 — Entrenamiento del Segmentador Vascular

Comparación de U-Net y U-Net++ con encoder ResNet-34 preentrenado (ImageNet).
Transfer learning + augmentación específica para microcirculación.

In [ ]:
# pip install segmentation-models-pytorch albumentations torch torchvision
import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH   = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))
DATASET_DIR = DATA_PATH / "dataset_nnunet"
MODEL_DIR   = DATA_PATH / "models"
MODEL_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")


## 4.1 Dataset y augmentación

In [ ]:
TRAIN_AUG = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=20, p=0.5),
    # Específicos para microcirculación
    A.RandomBrightnessContrast(brightness_limit=0.2,
                               contrast_limit=0.2, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(p=0.2),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.ElasticTransform(alpha=50, sigma=5, p=0.2),  # deformación de tejido
    A.Normalize(mean=(0.485,), std=(0.229,)),        # ImageNet 1-canal
    ToTensorV2()
])

VAL_AUG = A.Compose([
    A.Normalize(mean=(0.485,), std=(0.229,)),
    ToTensorV2()
])


class VesselDataset(Dataset):
    def __init__(self, img_dir: Path, mask_dir: Path,
                 transform=None, img_size: int = 512):
        self.imgs      = sorted(img_dir.glob("*.png"))
        self.mask_dir  = mask_dir
        self.transform = transform
        self.img_size  = img_size

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img_path  = self.imgs[idx]
        mask_path = self.mask_dir / img_path.name

        img  = cv2.imread(str(img_path),  cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        img  = cv2.resize(img,  (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size),
                          interpolation=cv2.INTER_NEAREST)

        # Expandir canal para albumentations
        img  = np.stack([img, img, img], axis=-1)  # 3 canales para ResNet
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']
            mask = aug['mask'].unsqueeze(0)

        return img, mask


## 4.2 Definir modelos

In [ ]:
def build_model(arch: str = "unet", encoder: str = "resnet34") -> nn.Module:
    """
    arch: 'unet' | 'unetplusplus'
    encoder: 'resnet34' | 'resnet50' | 'efficientnet-b0'
    """
    kwargs = dict(
        encoder_name    = encoder,
        encoder_weights = "imagenet",   # transfer learning
        in_channels     = 3,
        classes         = 1,
        activation      = None          # aplicamos sigmoid en la pérdida
    )
    if arch == "unet":
        return smp.Unet(**kwargs)
    elif arch == "unetplusplus":
        return smp.UnetPlusPlus(**kwargs)
    else:
        raise ValueError(f"Arquitectura desconocida: {arch}")

model_unet   = build_model("unet",        "resnet34").to(DEVICE)
model_unetpp = build_model("unetplusplus", "resnet34").to(DEVICE)

total_params = sum(p.numel() for p in model_unet.parameters()) / 1e6
print(f"U-Net parámetros: {total_params:.1f}M")


## 4.3 Función de pérdida — DiceBCE

In [ ]:
class DiceBCELoss(nn.Module):
    """Combinación de Dice + BCE — estándar para segmentación vascular."""
    def __init__(self, smooth=1.0, dice_weight=0.5):
        super().__init__()
        self.smooth      = smooth
        self.dice_weight = dice_weight
        self.bce         = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss  = self.bce(logits, targets)
        probs     = torch.sigmoid(logits)
        inter     = (probs * targets).sum(dim=(2, 3))
        dice_loss = 1 - (2 * inter + self.smooth) / (
            probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth)
        return self.dice_weight * dice_loss.mean() + (1 - self.dice_weight) * bce_loss


def dice_score(logits, targets, threshold=0.5) -> float:
    probs = (torch.sigmoid(logits) > threshold).float()
    inter = (probs * targets).sum(dim=(2, 3))
    return ((2 * inter + 1) / (probs.sum(dim=(2, 3)) +
             targets.sum(dim=(2, 3)) + 1)).mean().item()


## 4.4 Loop de entrenamiento

In [ ]:
def train_model(model, train_loader, val_loader,
                epochs=50, lr=1e-4, model_name="model") -> dict:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6)
    criterion = DiceBCELoss()

    history   = {"train_loss": [], "val_loss": [], "val_dice": []}
    best_dice = 0.0
    best_path = MODEL_DIR / f"{model_name}_best.pth"

    for epoch in range(1, epochs + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, masks)
            loss.backward(); optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validation ─────────────────────────────────────────────────────
        model.eval()
        val_loss, val_dice = 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, masks).item()
                val_dice += dice_score(logits, masks)
        val_loss /= len(val_loader)
        val_dice /= len(val_loader)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dice"].append(val_dice)

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), best_path)

        if epoch % 10 == 0:
            print(f"Ep {epoch:3d}/{epochs} | "
                  f"train={train_loss:.4f} val={val_loss:.4f} dice={val_dice:.4f}")

    print(f"Mejor Dice: {best_dice:.4f}  →  {best_path}")
    return history


# Instanciar datasets
train_ds = VesselDataset(DATASET_DIR/"imagesTr", DATASET_DIR/"labelsTr",
                         transform=TRAIN_AUG)
val_ds   = VesselDataset(DATASET_DIR/"imagesTs", DATASET_DIR/"labelsTr",
                         transform=VAL_AUG)

train_loader = DataLoader(train_ds, batch_size=4,  shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=4,  shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")


## 4.5 Entrenar y comparar U-Net vs U-Net++

In [ ]:
# Entrenar U-Net base
print("=== U-Net ===")
hist_unet = train_model(model_unet, train_loader, val_loader,
                        epochs=50, lr=1e-4, model_name="unet_r34")

# Entrenar U-Net++
print("\n=== U-Net++ ===")
hist_unetpp = train_model(model_unetpp, train_loader, val_loader,
                          epochs=50, lr=1e-4, model_name="unetpp_r34")


## 4.6 Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, ylabel in zip(axes, ["val_loss","val_dice"],
                            ["Validation Loss","Validation Dice"]):
    ax.plot(hist_unet[key],   label="U-Net")
    ax.plot(hist_unetpp[key], label="U-Net++")
    ax.set_xlabel("Época"); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("Comparación U-Net vs U-Net++", fontsize=13)
plt.tight_layout()
plt.savefig(DATA_PATH / "fig_04_training_curves.png", dpi=150)
plt.show()

best_unet   = max(hist_unet['val_dice'])
best_unetpp = max(hist_unetpp['val_dice'])
print(f"Mejor Dice U-Net:    {best_unet:.4f}")
print(f"Mejor Dice U-Net++:  {best_unetpp:.4f}")
print(f"\nArquitectura ganadora: {'U-Net++' if best_unetpp > best_unet else 'U-Net'}")
